# 13 · TCN-BiLSTM Block

**The question:** Does adding a **TCN** (dilated convolutions) and a **BiLSTM** after the fusion improve the network?

| | |
|---|---|
| **Input** | Scaled `total_and_byte44` inputs; weighting (Notebook 10), convolution (Notebook 11) and fusion (Notebook 12) chosen so far |
| **What we do** | Train the network with and without the TCN-BiLSTM block, 3 seeds each |
| **Output** | A decision: keep the block or not. Notebook 14 assembles the proposed model |

**An honest note before we start.** TCNs and BiLSTMs were invented for **sequences in time**. Our rows are single completed flows with no timestamps
(Notebook 01), so here the block runs over the **sequence of feature tokens**, not over time. It can still learn relations between distant features,
but it cannot learn anything about the order of events. Notebook 08 already found that a BiLSTM alone added nothing; this notebook tests the fuller
TCN-BiLSTM block under the improved setup, so the paper can report a fair answer.

**Decision rule (same as Notebooks 11-12):** keep the block only if it raises the 3-seed mean macro F1 and is better in at least 2 of the 3 seeds.

## 1. Setup

In [1]:
import sys
import warnings
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score

from src.notebook_helpers import TRAIN_CSV, RAW_SHA256, CLASSES, MODELS, sha256, save_table, save_figure, save_json, read_json, setup_style, write_leaderboard
from src.flow_networks import FlowData, BASE_CONFIG, TCNBiLSTM, run_config, paired_comparison, leaderboard_row, show_code, DEVICE
setup_style()
assert sha256(TRAIN_CSV) == RAW_SHA256['train']

WEIGHTING = read_json(MODELS / '10_rare_class_choice.json')['network_weighting']
previous = read_json(MODELS / '12_decision.json')
SCALES, FUSION = previous['scales'], previous['fusion']
print(f'weighting: {WEIGHTING} | scales: {SCALES} | fusion: {FUSION} | device: {DEVICE}')
data = FlowData('total_and_byte44')

weighting: sqrt | scales: [3, 5, 7] | fusion: cross | device: cuda


## 2. The block

- **TCN:** three convolutions with dilation 1, 2 and 4. Dilation means the window skips positions, so the third layer looks 4 tokens apart.
  Each layer adds to its input (a *residual* connection), so the original signal is never lost.
- **BiLSTM:** reads the token sequence from left to right and from right to left, and combines both readings.

In [2]:
show_code(TCNBiLSTM)

```python
class TCNBiLSTM(nn.Module):
    """A temporal-convolution (TCN) block with growing dilation (1, 2, 4), followed by a bidirectional LSTM.
    On our data it runs over the fused feature-token sequence, not over time."""

    def __init__(self, channels=64):
        super().__init__()
        self.tcn = nn.ModuleList([nn.Conv1d(channels, channels, kernel_size=3, padding=d, dilation=d) for d in (1, 2, 4)])
        self.lstm = nn.LSTM(channels, channels // 2, batch_first=True, bidirectional=True)

    def forward(self, h):                                   # h: (batch, length, channels)
        x = h.transpose(1, 2)
        for conv in self.tcn:
            x = x + torch.relu(conv(x))                     # residual connection keeps the original signal
        out, _ = self.lstm(x.transpose(1, 2))
        return out

```

## 3. Train both versions (3 seeds each)

In [3]:
WITHOUT = dict(BASE_CONFIG, weighting=WEIGHTING, scales=SCALES, fusion=FUSION, tcn_bilstm=False)
WITH = dict(BASE_CONFIG, weighting=WEIGHTING, scales=SCALES, fusion=FUSION, tcn_bilstm=True)
without_runs, without_proba, _, _ = run_config(data, WITHOUT, label='without TCN-BiLSTM')
with_runs, with_proba, _, _ = run_config(data, WITH, label='with TCN-BiLSTM')

without TCN-BiLSTM seed 42: model_selection macro F1 0.6054 | stopping macro F1 0.6177


without TCN-BiLSTM seed 43: model_selection macro F1 0.5978 | stopping macro F1 0.6066


without TCN-BiLSTM seed 44: model_selection macro F1 0.5831 | stopping macro F1 0.5812


with TCN-BiLSTM seed 42: model_selection macro F1 0.6135 | stopping macro F1 0.6074


with TCN-BiLSTM seed 43: model_selection macro F1 0.5917 | stopping macro F1 0.5950


with TCN-BiLSTM seed 44: model_selection macro F1 0.6023 | stopping macro F1 0.6113


## 4. Results

In [4]:
runs = pd.concat([without_runs, with_runs], ignore_index=True)
save_table(runs, '13_runs')
display(runs[['experiment', 'seed', 'macro F1', 'accuracy', 'false-positive rate', 'Worms correct (of 13)', 'epochs', 'train minutes', 'parameters']])
display(runs.groupby('experiment', sort=False)[['macro F1', 'train minutes']].agg(['mean', 'std']))

y_eval = data.y['model_selection'].numpy()
comparison, overall = paired_comparison(y_eval, with_proba, without_proba, 'with vs without TCN-BiLSTM')
save_table(comparison, '13_comparison')
display(comparison)

per_class = pd.DataFrame({
    'without F1': np.mean([f1_score(y_eval, p.argmax(1), average=None, labels=range(10), zero_division=0) for p in without_proba.values()], axis=0),
    'with F1': np.mean([f1_score(y_eval, p.argmax(1), average=None, labels=range(10), zero_division=0) for p in with_proba.values()], axis=0),
}, index=CLASSES)
per_class['change'] = per_class.iloc[:, 1] - per_class.iloc[:, 0]
save_table(per_class, '13_per_class', index=True)
display(per_class.round(3))

,experiment,seed,macro F1,accuracy,false-positive rate,Worms correct (of 13),epochs,train minutes,parameters
0,without TCN-BiLSTM,42,0.6054,0.7939,0.1732,7,38,2.1216,85640
1,without TCN-BiLSTM,43,0.5978,0.7904,0.1809,8,35,1.9204,85640
2,without TCN-BiLSTM,44,0.5831,0.7911,0.1668,9,46,2.4518,85640
3,with TCN-BiLSTM,42,0.6135,0.7994,0.1461,9,39,2.7241,147784
4,with TCN-BiLSTM,43,0.5917,0.7934,0.1734,7,27,1.7930,147784
5,with TCN-BiLSTM,44,0.6023,0.7988,0.1484,7,34,2.3461,147784


macro F1        train minutes       
                       mean    std          mean    std
experiment                                             
without TCN-BiLSTM   0.5955 0.0113        2.1646 0.2683
with TCN-BiLSTM      0.6025 0.0109        2.2877 0.4683

,comparison,seed,difference,95% low,95% high,verdict
0,with vs without TCN-BiLSTM,42,0.0081,-0.0060,0.0238,could be chance
1,with vs without TCN-BiLSTM,43,-0.0061,-0.0168,0.0043,could be chance
2,with vs without TCN-BiLSTM,44,0.0191,0.0044,0.0332,first clearly better


,without F1,with F1,change
Analysis,0.2680,0.2760,0.0070
Backdoor,0.2310,0.2470,0.0160
DoS,0.4780,0.4820,0.0040
Exploits,0.6620,0.6670,0.0040
Fuzzers,0.7160,0.7250,0.0090
Generic,0.9880,0.9880,-0.0000
Normal,0.8990,0.9060,0.0070
Reconnaissance,0.8250,0.8240,-0.0010
Shellcode,0.5860,0.5980,0.0120
Worms,0.3000,0.3120,0.0130


**What we see:**

- **With TCN-BiLSTM the mean macro F1 rises from 0.596 to 0.603** and is higher for 2 of 3 seeds. The seed tests: chance for seeds 42 and 43,
  clearly better for seed 44. Again a small gain that is not reliable on its own.
- The improvement is broad but small: 8 of the 10 classes gain a little (largest: Backdoor +0.016, Worms +0.013, Shellcode +0.012).
- False alarms fall slightly (about 17.4% → 15.6% on average).
- The block nearly doubles the size (148k vs 86k parameters), yet training time stays similar because it stops earlier.
- This differs from Notebook 08, where a plain BiLSTM added nothing. Here the block includes the TCN with residual connections and sits on top of the
  weighted, cross-attention design. Its gain still comes from relating **feature tokens**, not from time order (there is none in these rows).

## 5. Decision

In [5]:
wins = int(sum(with_runs.set_index('seed')['macro F1'] > without_runs.set_index('seed')['macro F1']))
gain = with_runs['macro F1'].mean() - without_runs['macro F1'].mean()
keep = bool(gain > 0 and wins >= 2)
decision = {'block': 'TCN-BiLSTM', 'mean gain': round(float(gain), 4), 'seeds where better': f'{wins} of 3',
            'bootstrap verdict over seeds': overall, 'kept': keep, 'tcn_bilstm': keep, 'fusion': FUSION, 'scales': SCALES}
save_json(decision, MODELS / '13_decision.json')
display(pd.Series(decision).to_frame('decision'))
label = f'scales {SCALES}, {FUSION}-attention ({WEIGHTING})'
write_leaderboard('13', [leaderboard_row(f'FlowNet {label}', without_runs), leaderboard_row(f'FlowNet {label} + TCN-BiLSTM', with_runs)])
assert sha256(TRAIN_CSV) == RAW_SHA256['train']

,decision
block,TCN-BiLSTM
mean gain,0.0071
seeds where better,2 of 3
bootstrap verdict over seeds,depends on the seed
kept,True
tcn_bilstm,True
fusion,cross
scales,"[3, 5, 7]"


## Summary

### What we learned

| Question | Answer |
|---|---|
| Does TCN-BiLSTM help? | **Slightly**: +0.007 mean macro F1, better in 2 of 3 seeds, bootstrap clear only for one seed |
| Where? | Small gains spread over most classes; slightly fewer false alarms |
| What does it cost? | About 1.7× more parameters (148k vs 86k) |
| Decision (by the fixed rule) | **Kept** |

### The honest overall picture of Notebooks 11-13

All three blocks passed the rule, but each by a small margin that depends on the seed. Together they lift the network from 0.588 (Notebook 10, sqrt
weights) to 0.603. LightGBM with sqrt weights is at 0.666 on the same rows, so the network is still clearly behind.

### What comes next

**Notebook 14** assembles the proposed model from these decisions, gives it a small fair tuning, and compares it with all earlier models.